# 🧩 Notebook 1 — Corpus Dialog2Flow y carga de diálogos

En esta notebook construimos un DataFrame de turnos a partir del dataset **Dialog2Flow**.

La notebook:

1. descarga los datasets de Dialog2Flow desde Hugging Face;
2. recorre los diálogos completos;
3. genera una fila por turno;
4. permite definir una cantidad mínima de turnos mediante `CANTIDAD_VECTORES`;
5. guarda el resultado en `multiwoz_turns.pkl` para usarlo en las siguientes notebooks.

## 1️⃣ Instalación de dependencias

Dialog2Flow utiliza un script de carga propio en Hugging Face. Para evitar problemas con cambios recientes de la librería `datasets`, fijamos una versión compatible con `trust_remote_code=True`.

In [1]:
!pip install -q "datasets==2.21.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


## 2️⃣ Parámetros generales

`CANTIDAD_VECTORES` indica la cantidad mínima de turnos que queremos cargar.

El corte se realiza **después de completar el diálogo actual**, para no dejar conversaciones truncadas.

Si se quiere procesar todo el corpus, usar:

```python
CANTIDAD_VECTORES = None
```

In [2]:
REPO_ID = "sergioburdisso/dialog2flow-dataset"

# Cantidad mínima de turnos/vectores a cargar.
# La cantidad final puede ser un poco mayor porque no se cortan diálogos por la mitad.
CANTIDAD_VECTORES = 1000000

## 3️⃣ Datasets disponibles

Definimos manualmente la lista de datasets individuales incluidos en Dialog2Flow. Esto evita depender de `get_dataset_config_names`, que puede cambiar su comportamiento entre versiones de `datasets`.

In [3]:
DATASETS_DIALOG2FLOW = [
    "ABCD",
    "BiTOD",
    "Disambiguation",
    "DSTC2-Clean",
    "FRAMES",
    "GECOR",
    "HDSA-Dialog",
    "KETOD",
    "MS-DC",
    "MulDoGO",
    "MultiWOZ_2.1",
    "MULTIWOZ2_2",
    "SGD",
    "SimJointGEN",
    "SimJointMovie",
    "SimJointRestaurant",
    "Taskmaster1",
    "Taskmaster2",
    "Taskmaster3",
    "WOZ2_0",
]

print("Cantidad de datasets:", len(DATASETS_DIALOG2FLOW))
print(DATASETS_DIALOG2FLOW)

Cantidad de datasets: 20
['ABCD', 'BiTOD', 'Disambiguation', 'DSTC2-Clean', 'FRAMES', 'GECOR', 'HDSA-Dialog', 'KETOD', 'MS-DC', 'MulDoGO', 'MultiWOZ_2.1', 'MULTIWOZ2_2', 'SGD', 'SimJointGEN', 'SimJointMovie', 'SimJointRestaurant', 'Taskmaster1', 'Taskmaster2', 'Taskmaster3', 'WOZ2_0']


## 4️⃣ Carga de ejemplo

Antes de procesar todo, cargamos un dataset para revisar la estructura.

In [4]:
from datasets import load_dataset

config_ejemplo = "WOZ2_0"
dataset_ejemplo = load_dataset(REPO_ID, config_ejemplo, trust_remote_code=True)

dataset_ejemplo

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    test: Dataset({
        features: ['dialog'],
        num_rows: 400
    })
    train: Dataset({
        features: ['dialog'],
        num_rows: 600
    })
    validation: Dataset({
        features: ['dialog'],
        num_rows: 200
    })
})

In [5]:
primer_split = list(dataset_ejemplo.keys())[0]
primer_dialogo = dataset_ejemplo[primer_split][0]["dialog"]

print("Split:", primer_split)
print("Cantidad de turnos del primer diálogo:", len(primer_dialogo))
primer_dialogo[:2]

Split: test
Cantidad de turnos del primer diálogo: 1


[{'speaker': 'user',
  'text': 'What is the phone number and postcode of a cheap restaurant in the east part of town?',
  'domains': ['restaurant'],
  'labels': {'dialog_acts': {'acts': ['inform', 'request'],
    'main_acts': ['inform', 'request'],
    'original_acts': ['inform', 'request']},
   'slots': ['area', 'price range', 'phone', 'postcode'],
   'intents': None}}]

---

## 5️⃣ Construcción del DataFrame unificado de turnos

Recorremos los datasets, sus splits y sus diálogos. Para cada turno guardamos la información básica en una fila.

In [6]:
import pandas as pd

rows = []
stop = False
cantidad_dialogos = 0

for config in DATASETS_DIALOG2FLOW:
    if stop:
        break

    print("Procesando dataset:", config)
    dataset = load_dataset(REPO_ID, config, trust_remote_code=True)

    for split in dataset.keys():
        if stop:
            break

        print("  Split:", split)

        for i, item in enumerate(dataset[split]):
            dialog = item["dialog"]
            dialogue_id = f"{config}_{split}_{i}"

            for turn_id, turn in enumerate(dialog):
                labels = turn.get("labels", {}) or {}
                dialog_acts = labels.get("dialog_acts", {}) or {}

                rows.append({
                    "dataset": config,
                    "split": split,
                    "dialogue_id": dialogue_id,
                    "turn_id": turn_id,
                    "speaker": turn.get("speaker"),
                    "utterance": turn.get("text", ""),
                    "domains": turn.get("domains"),
                    "dialog_acts": dialog_acts.get("acts"),
                    "main_acts": dialog_acts.get("main_acts"),
                    "slots": labels.get("slots"),
                    "intents": labels.get("intents"),
                })

            cantidad_dialogos += 1

            # Cortamos solo después de terminar el diálogo actual.
            if CANTIDAD_VECTORES is not None and len(rows) >= CANTIDAD_VECTORES:
                stop = True
                break

    print("  Turnos acumulados:", len(rows))
    print("  Diálogos acumulados:", cantidad_dialogos)

print("Proceso terminado")
print("Total de turnos:", len(rows))
print("Total de diálogos:", cantidad_dialogos)

Procesando dataset: ABCD
  Split: test
  Split: train
  Split: validation
  Turnos acumulados: 19209
  Diálogos acumulados: 10041
Procesando dataset: BiTOD
  Split: test
  Split: train
  Split: validation
  Turnos acumulados: 88639
  Diálogos acumulados: 13730
Procesando dataset: Disambiguation
  Split: test
  Split: train
  Split: validation
  Turnos acumulados: 213625
  Diálogos acumulados: 24162
Procesando dataset: DSTC2-Clean
  Split: test
  Split: train
  Split: validation
  Turnos acumulados: 234249
  Diálogos acumulados: 27395
Procesando dataset: FRAMES
  Split: test
  Split: train
  Turnos acumulados: 252872
  Diálogos acumulados: 28763
Procesando dataset: GECOR
  Split: train
  Turnos acumulados: 255280
  Diálogos acumulados: 29439
Procesando dataset: HDSA-Dialog
  Split: test
  Split: train
  Split: validation
  Turnos acumulados: 345866
  Diálogos acumulados: 39456
Procesando dataset: KETOD
  Split: test
  Split: train
  Split: validation
  Turnos acumulados: 449061
  Diálog

In [7]:
df = pd.DataFrame(rows)

df.head()

,dataset,split,dialogue_id,turn_id,speaker,utterance,domains,dialog_acts,main_acts,slots,intents
0,ABCD,test,ABCD_test_0,0,user,Hi. My name is Chloe Zhang. I am curious as ...,[storewide query],None,None,None,[timing]
1,ABCD,test,ABCD_test_1,0,user,Hello. I recently signed up for a subscription...,[subscription inquiry],None,None,None,[manage dispute bill]
2,ABCD,test,ABCD_test_1,1,user,"sure, it's Albert Sanders and my account id is...",[subscription inquiry],None,None,[account id],None
3,ABCD,test,ABCD_test_1,2,user,yes its 7149958247,[subscription inquiry],None,None,[order id],None
4,ABCD,test,ABCD_test_2,0,user,I'm thinking about buying an item but first i ...,[single item query],None,None,None,[shirt]


In [8]:
print("Shape:", df.shape)
print("Cantidad de datasets incluidos:", df["dataset"].nunique())
print("Cantidad de diálogos incluidos:", df["dialogue_id"].nunique())
print("Cantidad de turnos incluidos:", len(df))

if CANTIDAD_VECTORES is not None:
    print("Cantidad mínima solicitada:", CANTIDAD_VECTORES)
    print("Excedente por completar el último diálogo:", len(df) - CANTIDAD_VECTORES)

Shape: (1000023, 11)
Cantidad de datasets incluidos: 13
Cantidad de diálogos incluidos: 101021
Cantidad de turnos incluidos: 1000023
Cantidad mínima solicitada: 1000000
Excedente por completar el último diálogo: 23


## 6️⃣ Guardado del DataFrame

Persistimos el DataFrame para utilizarlo en el resto del pipeline.

In [11]:
ARCHIVO_SALIDA = "dialogs-2.0.pkl"

In [12]:
df.to_pickle(ARCHIVO_SALIDA)

print("Archivo guardado:", ARCHIVO_SALIDA)

Archivo guardado: dialogs-2.0.pkl


In [13]:
from google.colab import files

files.download(ARCHIVO_SALIDA)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>